In [0]:
# import json

# from pyspark.sql import functions as F
# from pyspark.sql.window import Window


# # 1. Create a repeatable source fixture

# booking = {
#     "booking_id": "B001",
#     "passenger_id": "P001",
#     "booking_status": "CONFIRMED",
#     "created_at": "2026-11-14T07:55:00Z",
# }

# def assignment_event(payload, version, effective_at, emitted_at):
#     segment_id = payload["booking_segment_id"]
#     return {
#         "schema_version": 1,
#         "source_system": "booking_system",
#         "event_id": f"EVT_{segment_id}_V{version}",
#         "entity_type": "booking_segment",
#         "entity_id": segment_id,
#         "entity_version": version,
#         "effective_at": effective_at,
#         "emitted_at": emitted_at,
#         "payload": payload,
#     }

# incoming_events = [
#     assignment_event(
#         {
#             "booking_segment_id": "BS002",
#             "booking_id": "B002",
#             "original_flight_leg_id": "FL100_20261115",
#             "assigned_flight_leg_id": "FL100_20261115",
#             "assignment_status": "ASSIGNED",
#         },
#         1,
#         "2026-11-14T08:15:00Z",
#         "2026-11-14T08:16:00Z",
#     ),
#     assignment_event(
#         {
#             "booking_segment_id": "BS003",
#             "booking_id": "B003",
#             "original_flight_leg_id": "FL100_20261115",
#             "assigned_flight_leg_id": "FL100_20261115",
#             "assignment_status": "ASSIGNED",
#         },
#         1,
#         "2026-11-14T08:30:00Z",
#         "2026-11-14T08:31:00Z",
#     ),
#     assignment_event(
#         {
#             "booking_segment_id": "BS001",
#             "booking_id": booking["booking_id"],
#             "original_flight_leg_id": "FL100_20261115",
#             "assigned_flight_leg_id": "FL100_20261115",
#             "assignment_status": "ASSIGNED"
#         },
#         1,
#         "2026-11-14T08:00:00Z",
#         "2026-11-14T08:01:00Z"

#     ),
#     assignment_event(
#         {
#             "booking_segment_id": "BS001",
#             "booking_id": booking["booking_id"],
#             "original_flight_leg_id": "FL100_20261115",
#             "assigned_flight_leg_id": "FL200_20261115",
#             "assignment_status": "ASSIGNED"
#         },
#         2,
#         "2026-11-14T10:00:00Z",
#         "2026-11-14T10:01:00Z"

#     )
    
# ]





In [0]:
import json

from pyspark.sql import functions as F
from pyspark.sql.window import Window


# 1. Create a repeatable source fixture

booking = {
    "booking_id": "B001",
    "passenger_id": "P001",
    "booking_status": "CONFIRMED",
    "created_at": "2026-11-14T07:55:00Z",
}

booking_segment = {
    "booking_segment_id": "BS001",
    "booking_id": booking["booking_id"],
    "original_flight_leg_id": "FL100_20261115",
    "assigned_flight_leg_id": "FL100_20261115",
    "assignment_status": "ASSIGNED",
}

rebooked_segment = {
    **booking_segment,
    "assigned_flight_leg_id": "FL200_20261115",
}

def assignment_event(payload, version, effective_at, emitted_at):
    segment_id = payload["booking_segment_id"]
    return {
        "schema_version": 1,
        "source_system": "booking_system",
        "event_id": f"EVT_{segment_id}_V{version}",
        "entity_type": "booking_segment",
        "entity_id": segment_id,
        "entity_version": version,
        "effective_at": effective_at,
        "emitted_at": emitted_at,
        "payload": payload,
    }

initial_assignment_event = assignment_event(
    booking_segment, 1, "2026-11-14T08:00:00Z", "2026-11-14T08:01:00Z"
)
current_assignment_event = assignment_event(
    rebooked_segment, 2, "2026-11-14T10:00:00Z", "2026-11-14T10:01:00Z"
)

additional_assignment_events = [
    assignment_event(
        {
            "booking_segment_id": "BS002",
            "booking_id": "B002",
            "original_flight_leg_id": "FL100_20261115",
            "assigned_flight_leg_id": "FL100_20261115",
            "assignment_status": "ASSIGNED",
        },
        1,
        "2026-11-14T08:15:00Z",
        "2026-11-14T08:16:00Z",
    ),
    assignment_event(
        {
            "booking_segment_id": "BS003",
            "booking_id": "B003",
            "original_flight_leg_id": "FL100_20261115",
            "assigned_flight_leg_id": "FL100_20261115",
            "assignment_status": "ASSIGNED",
        },
        1,
        "2026-11-14T08:30:00Z",
        "2026-11-14T08:31:00Z",
    ),
]

incoming_events = [
    current_assignment_event,
    initial_assignment_event,
    current_assignment_event,
    *additional_assignment_events,
]